In [1]:
!pip install transformers

Defaulting to user installation because normal site-packages is not writeable


In [2]:
%pip install accelerate

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [3]:
%pip install pydantic

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [4]:
import json
import re
from typing import Optional
from pydantic import BaseModel, Field, ValidationError


# Define your expected output schema (customize fields to your task)
class ExtractionSchema(BaseModel):
  date: Optional[str] = Field(
      default=None, description="Extracted date or null"
  )
  location: Optional[str] = Field(
      default=None, description="Extracted location or null"
  )
  summary: Optional[str] = Field(
      default=None, description="Brief summary or null"
  )


def parse_and_validate_output(raw_output: str) -> dict:
  """Safely extracts JSON from raw LLM output and validates it against Pydantic."""
  json_str = None

  # 1. Try direct parsing
  try:
    data = json.loads(raw_output)
    if isinstance(data, dict):
      json_str = data
  except json.JSONDecodeError:
    pass

  # 2. Fallback: Search for JSON block via regex if direct parsing failed
  if not json_str:
    match = re.search(r"(\{.*?\})", raw_output, re.DOTALL)
    if match:
      try:
        data = json.loads(match.group(1))
        if isinstance(data, dict):
          json_str = data
      except json.JSONDecodeError:
        pass

  # 3. If we found a dictionary, validate with Pydantic
  if json_str:
    try:
      validated = ExtractionSchema(**json_str)
      return validated.dict(), True  # Success
    except ValidationError:
      # Schema mismatch, but we got JSON
      return json_str, False

  # 4. Total failure fallback
  return {"date": None, "location": None, "summary": None}, False

# 07 - Metadata extraction with EuroLLM

Run from the `notebooks/` folder. Install once: `%pip install transformers accelerate torch`.

Steps: (1) make a gold-standard template you verify by hand, (2) read the first pages of each PDF, (3) ask EuroLLM for JSON, (4) check the answers against the page text and against your gold rows.

Everything runs locally. Outputs go to `../output/extraction/`.

## 1. Settings

In [5]:
from pathlib import Path
import json, re, statistics, unicodedata, difflib
import pandas as pd
import pymupdf
from IPython.display import display

PDF_DIR = Path('../pdfs').resolve()
INVENTORY_DIR = Path('../output/inventory').resolve()
OUT_DIR = Path('../output/extraction').resolve()
OUT_DIR.mkdir(parents=True, exist_ok=True)


MODEL_ID = MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'
RUN_SCOPE = 'dev'        
FIRST_PAGES = 3              
MAX_INPUT_TOKENS = 8000      
FIELDS = ['title', 'authors', 'doi', 'publication_year', 'document_type', 'language', 'keywords']
LIST_FIELDS = {'authors', 'keywords'}
DOC_TYPES = ['journal_article', 'report', 'thesis', 'conference_paper', 'book_chapter', 'policy_brief', 'other']

read = lambda name: pd.read_csv(INVENTORY_DIR / name, dtype=str, keep_default_na=False, encoding='utf-8-sig')
documents, observations = read('documents.csv'), read('metadata_observations.csv')
print(len(documents), 'documents')

93 documents


## 2. Gold-standard template

Creates `gold_metadata.csv` once (never overwrites your edits): 40 documents, 25 `dev` and 15 `test`, pre-filled with the PDF-embedded candidates. **Verify every value against the PDF or the Publinova record.**

Fill-in rules: `;` between authors/keywords · `publication_year` as 4 digits · `document_type` from `journal_article, report, thesis, conference_paper, book_chapter, policy_brief, other` · `language` is `en`, `nl` or `mixed` · empty = not checked · `NONE` = checked and absent. Don't tune on `test` until the end.

In [6]:
GOLD_PATH = OUT_DIR / 'gold_metadata.csv'
if GOLD_PATH.exists():
    print('gold_metadata.csv exists; left untouched.')
else:
    pool = documents[documents.text_status.isin(['extractable', 'partial'])]
    pick = pool.sample(n=min(40, len(pool)), random_state=42)
    cand = lambda d, f: ' | '.join(observations[(observations.document_id == d) & (observations.field_name == f)].observed_value.unique())
    gold = pd.DataFrame({
        'document_id': pick.document_id, 'file_name': pick.file_name,
        'split': ['dev'] * min(25, len(pick)) + ['test'] * max(len(pick) - 25, 0),
        'title': [cand(d, 'title') for d in pick.document_id],
        'authors': [cand(d, 'author') for d in pick.document_id],
        'doi': [cand(d, 'doi') for d in pick.document_id],
        'publication_year': '', 'document_type': '', 'language': '',
        'keywords': [cand(d, 'pdf_keywords') for d in pick.document_id]})
    gold.to_csv(GOLD_PATH, index=False, encoding='utf-8-sig')
    print('Created', GOLD_PATH, '- now verify it by hand.')

gold_metadata.csv exists; left untouched.


## 3. Input and prompt

Each line of the first pages is read with its font size. Lines clearly larger than the page's typical text get a `[BIG]` marker, which helps the model find the title. The prompt ends with `{` so the model continues straight into a JSON object.

In [7]:
def page_text(page, header_pct=0.10, footer_pct=0.80):
    """Extract text with positional zones so the model can interpret structure."""
    H = page.rect.height
    W = page.rect.width

    zones = {'HEADER': [], 'LEFT': [], 'BODY': [], 'RIGHT': [], 'FOOTER': []}

    for block in page.get_text('dict', sort=True)['blocks']:
        if block.get('type') != 0:
            continue
        x0, y0, x1, y1 = block['bbox']
        lines = []
        for line in block['lines']:
            spans = [s for s in line['spans'] if s['text'].strip()]
            if spans:
                lines.append((max(s['size'] for s in spans),
                              ''.join(s['text'] for s in spans).strip()))
        if not lines:
            continue
        typical = statistics.median(s for s, _ in lines)
        text = '\n'.join(('[BIG] ' if s >= 1.2 * typical else '') + t for s, t in lines)

        cy = (y0 + y1) / 2
        cx = (x0 + x1) / 2

        if cy <= H * header_pct:
            zones['HEADER'].append(text)
        elif cy >= H * footer_pct:
            zones['FOOTER'].append(text)
        elif cx < W * 0.45:
            zones['LEFT'].append(text)
        elif cx > W * 0.55:
            zones['RIGHT'].append(text)
        else:
            zones['BODY'].append(text)

    parts = []
    for zone in ['HEADER', 'LEFT', 'BODY', 'RIGHT', 'FOOTER']:
        if zones[zone]:
            parts.append(f'[{zone}]\n' + '\n'.join(zones[zone]))
    return '\n\n'.join(parts)

def first_relevant_pages(pdf_path, min_chars=4500, max_pages=FIRST_PAGES):
    parts = []
    with pymupdf.open(PDF_DIR / pdf_path) as pdf:
        n = len(pdf)
        total = 0
        for i in range(min(max_pages, n)):
            block = f'=== PAGE {i+1} ===\n{page_text(pdf[i])}'
            parts.append(block)
            total += len(block)
            if total >= min_chars and i >= 0:
                break

        if n > 1:
            last = pdf[-1]
            H = last.rect.height
            footer_lines = []
            for b in last.get_text('dict', sort=True)['blocks']:
                if b.get('type') != 0:
                    continue
                if b['bbox'][1] >= H * 0.85:
                    for line in b['lines']:
                        s = ''.join(sp['text'] for sp in line['spans'] if sp['text'].strip())
                        if s.strip():
                            footer_lines.append(s.strip())
            if footer_lines:
                parts.append('=== LAST PAGE FOOTER ===\n' + '\n'.join(footer_lines))

    return '\n\n'.join(parts)

SYSTEM = (
    'You are a bibliographic metadata extractor for academic and professional documents '
    '(research papers, reports, presentations, whitepapers) in English or Dutch. '
    'You receive the first pages of a PDF, split into labelled zones:\n'
    '- [HEADER]: text at the top of a page (journal name, running title, page number)\n'
    '- [LEFT] / [RIGHT]: columns of a two-column layout\n'
    '- [BODY]: the main text\n'
    '- [FOOTER]: text at the bottom of a page (DOIs, copyright, dates, correspondence)\n'
    '- [BIG]: prefix on a line means it is in a larger font — usually the title\n'
    '- === PAGE N === and === LAST PAGE FOOTER === separate page sections\n\n'

    'Rules for each field:\n'
    '- title: the full main title. If multiple consecutive [BIG] lines belong together, '
    'JOIN them with a space. Never return a section heading like "OVER HET ONDERZOEK", '
    '"Abstract", "Introduction", or "No Title Given". If no title is printed, return null.\n'
    '- authors: a LIST of PLAIN STRINGS, e.g. ["John Smith", "Jane Doe"]. '
    'NEVER return objects like {"name": "John Smith"}. Split multiple authors by semicolon or "and".\n'
    '- doi: DOI may appear ANYWHERE — [HEADER], [FOOTER], [BODY], or the last page footer. '
    'Look for "10.xxxx/yyyy", "doi:", or "https://doi.org/". Return only the DOI string '
    'without the URL prefix. If no DOI is printed anywhere, return null.\n'
    '- publication_year: 4-digit integer from "Published", "Received/Accepted", '
    'copyright line, or journal issue year. If none, return null.\n'
    '- document_type: exactly one of: ' + ', '.join(DOC_TYPES) + '.\n'
    '- language: "en", "nl", or "mixed".\n'
    '- keywords: list of strings from an explicit keywords section only. Empty list otherwise.\n\n'

    'Respond with ONE JSON object and NOTHING else. No markdown fences, no explanations, '
    'no text before or after. Do not repeat these instructions.\n'
    'Example:\n'
    '{"title": "The Example Study", "authors": ["A. Smith", "B. Jones"], '
    '"doi": "10.1234/example", "publication_year": 2024, '
    '"document_type": "journal_article", "language": "en", "keywords": ["alpha", "beta"]}'

    'IMPORTANT — DOI recognition:\n'
'The paper\'s OWN DOI is often printed in a citation line at the top of page 1, in formats like:\n'
'  "Transforming Water 2019, 11, 2643; doi:10.3390/w11122643"\n'
'  "Journal Name 2023, 5, 123; https://doi.org/10.1234/xyz"\n'
'  "DOI: 10.1234/xyz"\n'
'In these citation lines, the 10.xxxx/yyyy part IS the paper\'s own DOI. Extract it.\n'
'This line is NOT a reference. It may appear right after a reference marker like [13], '
'or on its own near the top of page 1. It is still the paper\'s own DOI.\n'
'Do NOT extract DOIs that appear inside "[N] Author, Journal..." reference entries in the '
'body or at the end of the paper. Only the citation line near the title/abstract counts.\n'
'If multiple DOIs appear, prefer the one in the top citation line or near the abstract.\n'
'If no DOI appears there, return null.\n\n'

'IMPORTANT — Keywords recognition:\n'
'Keywords appear immediately after the abstract, in one of these forms:\n'
'  "Keywords: alpha; beta; gamma"\n'
'  "Key words: alpha, beta, gamma"\n'
'  "Trefwoorden: alpha; beta; gamma"\n'
'  "Index Terms: alpha; beta"\n'
'Extract every term after the colon and before the next section heading (e.g., "1. Introduction").\n'
'If no keywords line exists, return [].\n\n'
)

def user_prompt(text):
    return (
        'Extract the metadata from the document below and return it as a single JSON object.\n\n'
        'DOCUMENT:\n'
        f'{text}'
    )

demo = documents[documents.text_status != 'no_text'].iloc[0]
print(demo.file_name); print(first_relevant_pages(demo.pdf_path)[:1500])

001_adaptive-capacities-for-diversified-flood-risk-management-strategies-learning-from-pilot-projects.pdf
=== PAGE 1 ===
[HEADER]
water

[LEFT]
Article
[BIG] Adaptive Capacities for Diversiﬁed Flood Risk
Flavia Simona Cosoveanu *, Jean-Marie Buijs *, Marloes Bakker and Teun Terpstra
Received:17 October 2019; Accepted:10 December 2019; Published:14 December 2019
1.Introduction

[BODY]
Management Strategies:Learning from Pilot Projects
Department of Technology, Water & Environment; HZ University of Applied Sciences, Het Groene Woud 1,
4331 NB Middelburg, The Netherlands; m.h.n.bakker@hz.nl (M.B.); t.terpstra@hz.nl (T.T.)
*Correspondence:coso0002@hz.nl (F.S.C.); jm.buijs@hz.nl (J.-M.B.)
Abstract:Diversiﬁcation of ﬂood risk management strategies (FRMS) in response to climate change
reliesontheadaptivecapacitiesofinstitutions.Althoughadaptivecapacitiesenableﬂexibility
and adjustment, more empirical research is needed to better grasp the role of adaptive capacities
toaccommodateexpectedclima

## 4. Load EuroLLM and run

In [8]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16)
model = model.to("cpu")
model.eval()

def extract(text):
    messages = [{'role': 'system', 'content': SYSTEM}, {'role': 'user', 'content': user_prompt(text)}]
    prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(
    prompt,
    return_tensors='pt',
    truncation=True,
    max_length=MAX_INPUT_TOKENS,
    add_special_tokens=False,
    ).to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=400,
            do_sample=False,
            pad_token_id=tok.pad_token_id or tok.eos_token_id,
        )
    return tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

/var/data/python/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 338/338 [00:00<00:00, 4896.88it/s]


In [9]:
RAW_PATH = OUT_DIR / 'eurollm_raw.jsonl'
runnable = documents[documents.text_status.isin(['extractable', 'partial'])]
if RUN_SCOPE == 'sample5':
    todo = runnable.head(5)
elif RUN_SCOPE == 'all':
    todo = runnable
else:
    g = pd.read_csv(GOLD_PATH, dtype=str, keep_default_na=False, encoding='utf-8-sig')
    todo = runnable[runnable.document_id.isin(g.loc[g.split == RUN_SCOPE, 'document_id'])]

done = {}
if RAW_PATH.exists():
    for line in RAW_PATH.read_text(encoding='utf-8').splitlines():
        r = json.loads(line); done[(r['document_id'], r['model'])] = r['raw']

with RAW_PATH.open('a', encoding='utf-8') as sink:
    for n, row in enumerate(todo.itertuples(index=False), start=1):
        if (row.document_id, MODEL_ID) in done:
            continue
        raw = extract(first_relevant_pages(row.pdf_path))
        done[(row.document_id, MODEL_ID)] = raw
        sink.write(json.dumps({'document_id': row.document_id, 'model': MODEL_ID, 'raw': raw}, ensure_ascii=False) + '\n')
        sink.flush()
        print(f'[{n}/{len(todo)}] {row.file_name[:70]}')

[1/25] 001_adaptive-capacities-for-diversified-flood-risk-management-strategi
[2/25] 005_building-relationships-in-higher-education-to-support-students-mot
[3/25] 011_fine-tuning-the-flavor-of-tetraselmis-chuii-by-modifying-nitrogen-
[4/25] 013_grip-op-teksten-grip-op-kennis.pdf
[5/25] 016_impacts-of-counting-protocols-for-light-microscopy-on-estimates-of
[6/25] 019_lezen-en-schrijven-in-natuur-en-techniekonderwijs.pdf
[7/25] 023_relationship-quality-in-higher-education-and-the-interplay-with-st
[8/25] 027_the-use-of-tetraselmis-chuii-as-seafood-flavoring-agent-in-a-veget
[9/25] 029_transition-towards-integrated-flood-risk-management-strategies.pdf
[10/25] 031_vakdidactische-kennisclips-over-tekststructuur-een-verkennende-stu
[11/25] 034_animal-burrows-in-your-levees.pdf
[12/25] 036_exploring-the-archetypes-of-sustainable-business-models-in-the-tou
[13/25] 040_autonoom-varen-in-zeeland.pdf
[14/25] 041_beleving-van-het-landschap-rond-zeedijken-in-zeeland.pdf
[15/25] 045_circular-tourism

## 5. Parse and sanity-check

Parsing never crashes on bad JSON; it records `parse_ok = False`. Titles, authors and DOIs are also checked against the page text: `grounded_* = False` means the model wrote something that is not actually on the first pages, which usually means a hallucination.

In [15]:
# --- Recovery cell: rebuild state without re-running Part 4 ---
from pathlib import Path
import json, re, unicodedata, difflib
import pandas as pd
import pymupdf
from IPython.display import display

PDF_DIR        = Path('../pdfs').resolve()
INVENTORY_DIR  = Path('../output/inventory').resolve()
OUT_DIR        = Path('../output/extraction').resolve()
GOLD_PATH      = OUT_DIR / 'gold_metadata.csv'
RAW_PATH       = OUT_DIR / 'eurollm_raw.jsonl'

MODEL_ID        = 'Qwen/Qwen2.5-1.5B-Instruct'
RUN_SCOPE       = 'dev'
FIELDS          = ['title','authors','doi','publication_year','document_type','language','keywords']
LIST_FIELDS     = {'authors','keywords'}
DOC_TYPES       = ['journal_article','report','thesis','conference_paper','book_chapter','policy_brief','other']

read = lambda name: pd.read_csv(INVENTORY_DIR / name, dtype=str, keep_default_na=False, encoding='utf-8-sig')
documents = read('documents.csv')

# Rebuild todo exactly as Part 4 would have
runnable = documents[documents.text_status.isin(['extractable','partial'])]
g = pd.read_csv(GOLD_PATH, dtype=str, keep_default_na=False, encoding='utf-8-sig')
todo = runnable[runnable.document_id.isin(g.loc[g.split == RUN_SCOPE, 'document_id'])]

# Reload raw LLM outputs saved by Part 4
done = {}
if RAW_PATH.exists():
    for line in RAW_PATH.read_text(encoding='utf-8').splitlines():
        r = json.loads(line)
        done[(r['document_id'], r['model'])] = r['raw']

print(f'documents: {len(documents)}')
print(f'todo (dev split): {len(todo)}')
print(f'done (raw outputs loaded): {len(done)}')

documents: 93
todo (dev split): 25
done (raw outputs loaded): 25


In [16]:
# --- Part 5: pure LLM extraction (no regex, no verification) ---

def squash(s):
    s = unicodedata.normalize('NFKD', str(s)).lower()
    return re.sub(r'[^a-z0-9]+', '', ''.join(c for c in s if not unicodedata.combining(c)))

def clean_doi(v):
    return re.sub(r'(?i)^(https?://(dx\.)?doi\.org/|doi:\s*)', '', str(v).strip()).rstrip('.,;)').lower()

def as_list(v):
    if v is None or v == '': return []
    if isinstance(v, str): v = re.split(r'[;|]', v)
    return [str(x).strip() for x in v if str(x).strip()]

NULL_VALUES = {'', 'null', 'none', 'nan', 'undefined', 'n/a'}

rows = []
for row in todo.itertuples(index=False):
    raw = done.get((row.document_id, MODEL_ID))
    if raw is None:
        continue

    rec = {'document_id': row.document_id, 'file_name': row.file_name, 'parse_ok': True}
    try:
        data, _ = json.JSONDecoder().raw_decode(raw)
    except Exception:
        data, rec['parse_ok'] = {}, False

    for f in FIELDS:
        v = data.get(f)

        if f in LIST_FIELDS:
            rec[f] = '; '.join(as_list(v))
            continue

        # Normalise null-like values to empty string
        if v is None or str(v).strip().lower() in NULL_VALUES:
            rec[f] = ''
        elif f == 'doi':
            rec[f] = clean_doi(v)
        else:
            rec[f] = str(v).strip()

    rows.append(rec)

extractions = pd.DataFrame(rows)
extractions.to_csv(OUT_DIR / 'eurollm_extractions.csv', index=False, encoding='utf-8-sig')

display(extractions[['file_name', 'title', 'authors', 'doi',
                     'publication_year', 'document_type', 'language']])

,file_name,title,authors,doi,publication_year,document_type,language
0,001_adaptive-capacities-for-diversified-flood-...,Adaptive Capacities for Diversified Flood Risk...,"Cosoveanu, Flavia Simona; Buijs, Jean-Marie; B...",,2019,conference_paper,en
1,005_building-relationships-in-higher-education...,Building relationships in higher education to ...,Martijn J. M. Leenknecht; Ingrid Snijders; Lis...,,2023,journal_article,en
2,011_fine-tuning-the-flavor-of-tetraselmis-chui...,Fine-tuning the flavor of Tetraselmis chuii by...,Bert Coleman; Christof Van Poucke; Bavo De Wit...,,2023,journal_article,en
3,013_grip-op-teksten-grip-op-kennis.pdf,"Grip on Texts, Grip on Knowledge",Daniela Polišenská; Joep Schins; Helge Bonset,,2025,conference_paper,en
4,016_impacts-of-counting-protocols-for-light-mi...,Impacts of counting protocols for light micros...,Pascalle Jacobs; Léon Serre-Fredj; Reinoud P. ...,10.1002/lom3.10651,2024,conference_paper,en
5,019_lezen-en-schrijven-in-natuur-en-techniekon...,Lezen en schrijven in Natuur- en Techniekonder...,Hilde Kooiker-den Boer; Ted Sanders; Jacquelin...,,2025,conference_paper,nl
6,023_relationship-quality-in-higher-education-a...,Relationship Quality in Higher Education and t...,Ingrid Snijders; Lisette Wijnia; Rebecca M. Ku...,10.1111/bjep.12455,2021,journal_article,en
7,027_the-use-of-tetraselmis-chuii-as-seafood-fl...,The Use of Tetraselmis chuii as Seafood Flavor...,Bert Coelho; Christof Van Poucke; Bavo De Witt...,10.1016/j.algal.2024.103538,2024,journal_article,en
8,029_transition-towards-integrated-flood-risk-m...,Transition Towards Integrated Flood Risk Manag...,Jean-Marie Buijs; Flavia Simona Cosoveanu; Mar...,,2020,conference_paper,en
9,031_vakdidactische-kennisclips-over-tekststruc...,Vakdidactische kennisclips over tekststructuur,H.S. Kooiker-den Boer; S.T.M. Bogaerds-Hazenbe...,10.59302/6fqk2y74,2024,conference_paper,en


## 6. Evaluate against the gold standard

Only gold cells that are filled in are scored. Title: similarity ≥ 0.90 counts as correct. Authors/keywords: set F1 (names compared as sorted tokens, so `Smith, J.` equals `J. Smith`). Everything else: exact match. Use `dev` while tuning; use `test` once at the end.

In [17]:
# --- Part 6: display extraction for manual review ---

for row in extractions.itertuples(index=False):
    print('=' * 90)
    print(row.file_name)
    print('-' * 90)
    for f in ['title', 'authors', 'doi', 'publication_year',
              'document_type', 'language', 'keywords']:
        print(f'{f:18s}: {getattr(row, f, "")}')
    print()

001_adaptive-capacities-for-diversified-flood-risk-management-strategies-learning-from-pilot-projects.pdf
------------------------------------------------------------------------------------------
title             : Adaptive Capacities for Diversified Flood Risk Management
authors           : Cosoveanu, Flavia Simona; Buijs, Jean-Marie; Bakker, Marloes; Terpstra, Teun
doi               : 
publication_year  : 2019
document_type     : conference_paper
language          : en
keywords          : 

005_building-relationships-in-higher-education-to-support-students-motivation.pdf
------------------------------------------------------------------------------------------
title             : Building relationships in higher education to support students' motivation
authors           : Martijn J. M. Leenknecht; Ingrid Snijders; Lisette Wijnia; Remy M. J. P. Rikers; Sofie M. M. Loyens
doi               : 
publication_year  : 2023
document_type     : journal_article
language          : en
keyword